# Prediction of Product Sales

- Author: Salam Odeh

## Load and Inspect Data

In [47]:
# Import libraries

import pandas as pd

import numpy as np

pd.set_option('display.max_columns', 100)

In [48]:
# Load the sales prediction dataset using Pandas

# Adjust this path to wherever you saved the CSV in your Drive

fpath = 'sales_predictions_2023.csv'

df = pd.read_csv(fpath)

df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [49]:
# Use .info() to preview datatypes and a summary of the DataFrame's columns

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   str    
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   str    
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   str    
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   str    
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   str    
 9   Outlet_Location_Type       8523 non-null   str    
 10  Outlet_Type                8523 non-null   str    
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 799.2 KB


### 1) How many rows and columns?

In [50]:
# .shape returns (n_rows, n_columns)

n_rows, n_cols = df.shape

print(f"Rows: {n_rows}")

print(f"Columns: {n_cols}")

Rows: 8523
Columns: 12


- The dataset has **8,523 rows** and **12 columns**.

### 2) What are the datatypes of each variable?

In [51]:
# .dtypes lists the datatype of every column

df.dtypes

Item_Identifier                  str
Item_Weight                  float64
Item_Fat_Content                 str
Item_Visibility              float64
Item_Type                        str
Item_MRP                     float64
Outlet_Identifier                str
Outlet_Establishment_Year      int64
Outlet_Size                      str
Outlet_Location_Type             str
Outlet_Type                      str
Item_Outlet_Sales            float64
dtype: object

- Numeric columns (`float64`/`int64`): `Item_Weight`, `Item_Visibility`, `Item_MRP`, `Outlet_Establishment_Year`, `Item_Outlet_Sales` (target)

- Categorical columns (`object`): `Item_Identifier`, `Item_Fat_Content`, `Item_Type`, `Outlet_Identifier`, `Outlet_Size`, `Outlet_Location_Type`, `Outlet_Type`

### 3) Are there duplicates? If so, drop any duplicates.

In [52]:
# Check for fully duplicated rows

print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 0


In [53]:
# Drop duplicates (if any). Even if 0, it is good practice to include this step.

df = df.drop_duplicates()



# Confirm shape after dropping duplicates

print(f"Shape after dropping duplicates: {df.shape}")

Shape after dropping duplicates: (8523, 12)


- There were **0 duplicate rows** in this dataset, so no rows were removed.

### 4) Identify missing values.

In [54]:
# Check for missing values in each column

df.isna().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

- `Item_Weight`: 1,463 missing values (numeric column)

- `Outlet_Size`: 2,410 missing values (categorical column)

- All other columns have 0 missing values.

### 5) Address the missing values by using a placeholder value.

In [55]:
# --- Item_Weight (numeric) ---

# Fill missing numeric values with a placeholder of 0

# (Note: for modeling later, we will use a proper SimpleImputer with

#  median strategy AFTER the train-test split to avoid data leakage.

#  For this initial EDA/cleaning pass, we use a simple placeholder

#  so our null-checking functions can report on this column.)

df['Item_Weight'] = df['Item_Weight'].fillna(0)



# --- Outlet_Size (categorical) ---

# Fill missing categorical values with the placeholder string 'MISSING'

df['Outlet_Size'] = df['Outlet_Size'].fillna('MISSING')



# Preview the changes

df[['Item_Weight', 'Outlet_Size']].isna().sum()

Item_Weight    0
Outlet_Size    0
dtype: int64

### 6) Confirm that there are no missing values after addressing them.

In [56]:
# Confirm there are no remaining missing values anywhere in the dataframe

df.isna().sum()

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

- All columns now show **0 missing values**.

### 7) Find and fix any inconsistent categories of data.

In [57]:
# Check each categorical column's unique values for inconsistencies

cat_cols = df.select_dtypes('object').columns



for col in cat_cols:

    print(f"--- {col} ---")

    print(df[col].unique())

    print()

--- Item_Identifier ---
<StringArray>
['FDA15', 'DRC01', 'FDN15', 'FDX07', 'NCD19', 'FDP36', 'FDO10', 'FDP10',
 'FDH17', 'FDU28',
 ...
 'FDB46', 'NCX17', 'FDH31', 'FDX13', 'NCU53', 'FDD28', 'FDU43', 'NCF55',
 'NCW30', 'NCW05']
Length: 1559, dtype: str

--- Item_Fat_Content ---
<StringArray>
['Low Fat', 'Regular', 'low fat', 'LF', 'reg']
Length: 5, dtype: str

--- Item_Type ---
<StringArray>
[                'Dairy',           'Soft Drinks',                  'Meat',
 'Fruits and Vegetables',             'Household',          'Baking Goods',
           'Snack Foods',          'Frozen Foods',             'Breakfast',
    'Health and Hygiene',           'Hard Drinks',                'Canned',
                'Breads',         'Starchy Foods',                'Others',
               'Seafood']
Length: 16, dtype: str

--- Outlet_Identifier ---
<StringArray>
['OUT049', 'OUT018', 'OUT010', 'OUT013', 'OUT027', 'OUT045', 'OUT017',
 'OUT046', 'OUT035', 'OUT019']
Length: 10, dtype: str

--- Outlet

C:\Users\salam\AppData\Local\Temp\ipykernel_22164\1666758094.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes('object').columns


- `Item_Fat_Content` has inconsistent categories representing the same two groups:

  - `'Low Fat'`, `'low fat'`, `'LF'` should all be **`'Low Fat'`**

  - `'Regular'`, `'reg'` should all be **`'Regular'`**

- All other categorical columns look consistent.

In [58]:
# Check value counts BEFORE fixing

df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [59]:
# Fix inconsistent categories using a mapping dictionary with .replace()

fat_content_map = {

    'low fat': 'Low Fat',

    'LF': 'Low Fat',

    'reg': 'Regular'

}



df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)



# Check value counts AFTER fixing - should now only be 2 categories

df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64

### 8) For any numerical columns, obtain the summary statistics of each (min, max, mean).

In [60]:
# .describe() returns summary statistics (count, mean, std, min, 25%, 50%, 75%, max)

# for all numeric columns

df.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
count,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,10.650590,0.066132,140.992782,1997.831867,2181.288914
std,6.431899,0.051598,62.275067,8.371760,1706.499616
min,0.000000,0.000000,31.290000,1985.000000,33.290000
25%,6.650000,0.026989,93.826500,1987.000000,834.247400
50%,11.000000,0.053931,143.012800,1999.000000,1794.331000
75%,16.000000,0.094585,185.643700,2004.000000,3101.296400
max,21.350000,0.328391,266.888400,2009.000000,13086.964800


In [61]:
# Focus on just min, max, and mean for a quick summary

df.describe().loc[['min', 'max', 'mean']]

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
min,0.00000,0.000000,31.290000,1985.000000,33.290000
max,21.35000,0.328391,266.888400,2009.000000,13086.964800
mean,10.65059,0.066132,140.992782,1997.831867,2181.288914


**Summary statistics observations:**



- `Item_Weight`: ranges from 0 (placeholder for missing) to ~21.35, mean ~11.85

- `Item_Visibility`: ranges from 0 to ~0.33, mean ~0.066 (the percentage of total display area)

- `Item_MRP`: ranges from ~31.29 to ~266.89, mean ~140.99 (price of the item)

- `Outlet_Establishment_Year`: ranges from 1985 to 2009

- `Item_Outlet_Sales` (target): ranges from ~33.29 to ~13,086.96, mean ~2,181.29

## Data Dictionary



| Variable Name | Description |

|---|---|

| Item_Identifier | Unique product ID |

| Item_Weight | Weight of product |

| Item_Fat_Content | Whether the product is low fat or regular |

| Item_Visibility | The percentage of total display area of all products in a store allocated to the particular product |

| Item_Type | The category to which the product belongs |

| Item_MRP | Maximum Retail Price (list price) of the product |

| Outlet_Identifier | Unique store ID |

| Outlet_Establishment_Year | The year in which store was established |

| Outlet_Size | The size of the store in terms of ground area covered |

| Outlet_Location_Type | The type of area in which the store is located |

| Outlet_Type | Whether the outlet is a grocery store or some sort of supermarket |

| Item_Outlet_Sales | Sales of the product in the particular store. This is the **target variable** to be predicted. |

## Clean Data



Summary of cleaning performed in this notebook:

1. Checked for and confirmed no duplicate rows.

2. Identified missing values in `Item_Weight` (1,463) and `Outlet_Size` (2,410).

3. Addressed missing values with placeholders (`0` for `Item_Weight`, `'MISSING'` for `Outlet_Size`) so EDA functions can report on null presence/frequency.

4. Fixed inconsistent categories in `Item_Fat_Content` (`low fat`/`LF` -> `Low Fat`, `reg` -> `Regular`).

5. Reviewed summary statistics (min/max/mean) for all numeric columns.



**Note:** In Part 5, we will reload the ORIGINAL (uncleaned) dataset from scratch and perform the train/test split BEFORE imputing missing values (using `SimpleImputer`), to avoid data leakage. The placeholder-based cleaning above is for this week's exploratory work only.